# L5 Evaluation Modern (评价)

原版 notebook 里最过时的部分主要是：

- `RetrievalQA`
- `QAGenerateChain`
- `QAEvalChain`

这个版本保留原来的教学目标：

1. 先搭一个可问答的 RAG 应用
2. 准备测试样本
3. 让系统回答
4. 再用另一个 LLM 进行评分

但整个流程改成了 LangChain 1.x 更常见的写法：
`Retriever + LCEL RAG chain + structured output generation + structured output grading`


In [23]:
import os
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv, find_dotenv
from pydantic import AliasChoices, BaseModel, Field, model_validator
from sentence_transformers import SentenceTransformer

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import CSVLoader

_ = load_dotenv(find_dotenv())

# 这一段和 L4-RAG-modern 基本同构：
# 我们先解决“如何稳定找到本地 embedding 模型”这个问题。
# 这样后面的评测 notebook 就不会依赖云端 embedding 服务。
def find_workspace_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if candidate.name == "AIAgent":
            return candidate
    raise FileNotFoundError("Could not find the AIAgent workspace root.")

def find_local_bge_snapshot() -> Path:
    workspace_root = find_workspace_root()
    snapshot_root = workspace_root / "models" / "models--BAAI--bge-small-en-v1.5" / "snapshots"

    # macOS 下 snapshots 目录里可能混入 .DS_Store 这类隐藏文件。
    # 这里显式只保留“真正存在 config.json 的模型目录”，避免误把隐藏文件当成模型路径。
    snapshots = sorted(
        path for path in snapshot_root.iterdir()
        if path.is_dir() and (path / "config.json").exists()
    )

    if not snapshots:
        raise FileNotFoundError(f"No valid model snapshot found under {snapshot_root}")

    return snapshots[0]

class LocalBGEEmbeddings(Embeddings):
    # 这个类的作用是“把本地 sentence-transformers 模型接成 LangChain Embeddings 接口”。
    # 只要实现 embed_documents 和 embed_query，向量库和 retriever 就都能复用这套模型。
    def __init__(self, model_path: str):
        self.model = SentenceTransformer(model_path)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        # 建库时会走这个方法：把很多文档一次性转成向量。
        vectors = self.model.encode(texts, normalize_embeddings=True)
        return vectors.tolist()

    def embed_query(self, text: str) -> list[float]:
        # 查询时会走这个方法：把“用户问题”转成一个查询向量。
        vector = self.model.encode(text, normalize_embeddings=True)
        return vector.tolist()

llm = ChatOpenAI(
    temperature=0.0,
    model="qwen-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)


In [24]:
file = "OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file, encoding="utf-8")
data = loader.load()

# 这一段先搭一个“被评估对象”出来，也就是我们的 QA / RAG 应用。
# 没有被评估对象，后面的样本生成和评分都无从谈起。
embeddings = LocalBGEEmbeddings(str(find_local_bge_snapshot()))
vectorstore = InMemoryVectorStore.from_documents(data, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(retrieved_docs: list[Document]) -> str:
    # 检索回来的文档要先被拼接成上下文文本，再交给大模型阅读。
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

qa_prompt = ChatPromptTemplate.from_template(
    '''
    You are a product catalog assistant.
    Answer the question only with information from the retrieved context.
    If the answer is not in the context, say you cannot answer from the catalog.

    Context:
    {context}

    Question:
    {question}
    '''
)

# qa_chain 就是我们要被评估的系统。
# 你可以把它看成一个函数：
# 输入 = 用户问题
# 中间 = 检索相关文档 + 拼 Prompt
# 输出 = 模型答案
qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)


In [25]:
# 先保留原教程里“手写几个测试样本”的思路。
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes, the Cozy Comfort Pullover Set has side pockets.",
    },
    {
        "query": "What materials are used in the Ultra-Loft Down Hoodie?",
        "answer": "I should answer using the materials listed in the catalog entry for the Ultra-Loft Down Hoodie.",
    },
]

examples


[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'answer': 'Yes, the Cozy Comfort Pullover Set has side pockets.'},
 {'query': 'What materials are used in the Ultra-Loft Down Hoodie?',
  'answer': 'I should answer using the materials listed in the catalog entry for the Ultra-Loft Down Hoodie.'}]

In [26]:
# 下面用 structured output 取代旧版 QAGenerateChain。
# 学习重点是：你完全可以自己定义“测试用例长什么样”，而不是依赖一个老的黑盒链。
class QAPair(BaseModel):
    # 一条评测样本最核心的就是“问题 + 参考答案”。
    query: str = Field(
        validation_alias=AliasChoices("query", "question"),
        serialization_alias="query",
        description="A realistic user question answerable from the document.",
    )
    answer: str = Field(description="The expected answer grounded in the document.")

class QAPairBatch(BaseModel):
    # 之所以再包一层 Batch，是因为 structured output 在返回列表时更稳定。
    qa_pairs: list[QAPair] = Field(description="One or more QA pairs generated from the document.")

    @model_validator(mode="before")
    @classmethod
    def wrap_single_pair(cls, value):
        # 有些模型会直接返回单个 {"query": ..., "answer": ...}。
        # 这里统一包装成 {"qa_pairs": [...]}，避免因为外层结构漂移报错。
        if isinstance(value, dict) and "qa_pairs" not in value and (
            "query" in value or "question" in value
        ) and "answer" in value:
            return {"qa_pairs": [value]}
        return value

example_generator_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Generate one grounded QA pair from the provided product document. "
            "Keep the answer faithful to the text. "
            "Use the exact field names 'query' and 'answer'. "
            "Return valid JSON structured data only."
        ),
        ("human", "Document text:\n{document_text}\n\nReturn JSON only."),
    ]
)

# with_structured_output 会强约束模型返回成我们定义的 Pydantic 结构。
# 这样我们就不用手写 JSON 解析，也不用担心字段名乱掉。
example_generator = (
    example_generator_prompt
    | llm.with_structured_output(QAPairBatch, method="json_mode")
)

generated_examples = []
for doc in data[:5]:
    # 对前 5 条文档逐条生成评测样本。
    # 这样做的本质是：让另一个 LLM 先帮我们“造题 + 给标准答案”。
    batch = example_generator.invoke({"document_text": doc.page_content})
    generated_examples.extend(pair.model_dump() for pair in batch.qa_pairs)

generated_examples[:2]


[{'query': "What is the weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz."},
 {'query': 'What are the dimensions of the Small and Medium Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'The dimensions for the Small size are 18" x 28", and for the Medium size, they are 22.5" x 34.5".'}]

In [27]:
# 合并手写样本和自动生成样本，形成一个更像“迷你评测集”的列表。
full_examples = examples + generated_examples
print(f"Total evaluation examples: {len(full_examples)}")
full_examples[:3]


Total evaluation examples: 7


[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'answer': 'Yes, the Cozy Comfort Pullover Set has side pockets.'},
 {'query': 'What materials are used in the Ultra-Loft Down Hoodie?',
  'answer': 'I should answer using the materials listed in the catalog entry for the Ultra-Loft Down Hoodie.'},
 {'query': "What is the weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz."}]

In [28]:
# 让 QA 系统逐题作答。
# 这一步是在“跑被评估模型”，产出 predictions。
predictions = []
for example in full_examples:
    # 每次只把 query 传给 qa_chain，系统内部会自己完成检索和生成。
    result = qa_chain.invoke(example["query"])
    predictions.append(
        {
            "query": example["query"],
            "result": result,
        }
    )

predictions[:2]


[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'result': 'Yes, the Cozy Comfort Pullover Set does have side pockets. The description mentions that the pull-on pants feature "side pockets and a modern slim leg."'},
 {'query': 'What materials are used in the Ultra-Loft Down Hoodie?',
  'result': 'The Ultra-Loft Down Hoodie, as per the catalog, is not explicitly listed. However, based on the context provided, there are several jackets with similar names and features. If you\'re referring to the "Ultra-Loft 850 Big Baffle Hooded Jacket," it uses a Pertex Y-shaped yarn technology for the shell, which is a tightly interlocked weave that resists moisture and keeps down feathers inside. It\'s filled with premium 850-fill-power goose down, treated with DownTek for water repellency.\n\nIf you meant another specific model, please provide the exact name, and I can give you the correct details from the catalog.'}]

In [29]:
# 如果你想人工抽查，最先应该做的是看单题检索结果和最终答案。
sample_question = full_examples[0]["query"]
sample_context = retriever.invoke(sample_question)
sample_answer = qa_chain.invoke(sample_question)

print("Question:", sample_question)
print("\nRetrieved documents:")
for index, doc in enumerate(sample_context, start=1):
    print(f"Document {index}:")
    print(doc.page_content[:500])
    print("-" * 80)

print("\nModel answer:")
print(sample_answer)


Question: Do the Cozy Comfort Pullover Set have side pockets?

Retrieved documents:
Document 1:
: 10
name: Cozy Comfort Pullover Set, Stripe
description: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.

Size & Fit
- Pants are Favorite Fit: Sits lower on the waist.
- Relaxed Fit: Our most generous fit sits farthest from the body.

Fabric & Care
- In the softest blend of 63% polyester, 35% rayon and 2% spandex.

Additional Features
- Relaxed fit top wi
--------------------------------------------------------------------------------
Document 2:
: 73
name: Cozy Cuddles Knit Pullover Set
description: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out. 

Size & Fit 
Pants are Favorite Fit: Sits lower on the waist. 
Relaxed F

In [30]:
# 下面用 structured output 取代旧版 QAEvalChain。
# 评分标准我们自己显式写出来，这样更容易理解、也更容易调整。
class GradeResult(BaseModel):
    # 这里用离散标签而不是自由文本，是为了让评测结果更容易统计。
    grade: Literal["CORRECT", "PARTIAL", "INCORRECT"] = Field(
        description="How well the prediction matches the reference answer."
    )
    reasoning: str = Field(description="Short explanation for the grade.")

    @model_validator(mode="before")
    @classmethod
    def normalize_alternative_grader_output(cls, value):
        # 有些模型会返回 is_correct / feedback，而不是 grade / reasoning。
        # 这里统一映射回 notebook 后续统计要用的标准字段。
        if isinstance(value, dict) and "grade" not in value:
            if "is_correct" in value:
                value = dict(value)
                is_correct = value.get("is_correct")
                feedback = value.get("feedback") or value.get("reasoning") or ""
                value["grade"] = "CORRECT" if is_correct is True else "INCORRECT"
                value["reasoning"] = feedback
        if isinstance(value, dict) and "grade" in value:
            value = dict(value)
            numeric_grade_map = {
                "0": "INCORRECT",
                "1": "PARTIAL",
                "2": "CORRECT",
                0: "INCORRECT",
                1: "PARTIAL",
                2: "CORRECT",
            }
            value["grade"] = numeric_grade_map.get(value["grade"], value["grade"])
        return value

grader_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are grading answers for a retrieval QA system. "
            "Compare the model answer against the reference answer. "
            "Use the exact field names 'grade' and 'reasoning'. "
            "The grade must be one of: CORRECT, PARTIAL, INCORRECT. "
            "Return valid JSON structured data only."
        ),
        (
            "human",
            "Question: {query}\n\n"
            "Reference answer: {reference_answer}\n\n"
            "Model answer: {model_answer}\n\nReturn JSON only."
        ),
    ]
)

# 这个 grader_chain 本质上就是“LLM 裁判”。
# 它不负责回答问题，只负责比较：
# 标准答案 vs 系统预测答案
# 然后输出一个结构化判定结果。
grader_chain = grader_prompt | llm.with_structured_output(GradeResult, method="json_mode")

graded_outputs = []
for example, prediction in zip(full_examples, predictions):
    grade = grader_chain.invoke(
        {
            "query": example["query"],
            "reference_answer": example["answer"],
            "model_answer": prediction["result"],
        }
    )
    graded_outputs.append(grade)

graded_outputs[:2]


[GradeResult(grade='CORRECT', reasoning="The model answer correctly states that the Cozy Comfort Pullover Set has side pockets, which aligns with the reference answer. The additional details about the pants' features do not contradict the core information and provide extra context."),
 GradeResult(grade='PARTIAL', reasoning="The model answer provides detailed information about a similar jacket, the 'Ultra-Loft 850 Big Baffle Hooded Jacket,' but it does not directly answer the question about the 'Ultra-Loft Down Hoodie.' The materials for the specific hoodie in question are not provided, and the answer is based on an assumption that the user might be referring to a different, though similar, product. This makes the response partially correct as it gives useful information but does not fully address the original question.")]

In [31]:
# 最后把评估结果打印出来，形成一个最小可读的评估报告。
for index, (example, prediction, grade) in enumerate(zip(full_examples, predictions, graded_outputs), start=1):
    print(f"Example {index}")
    print("Question:", example["query"])
    print("Reference:", example["answer"])
    print("Prediction:", prediction["result"])
    print("Grade:", grade.grade)
    print("Reasoning:", grade.reasoning)
    print("=" * 100)


Example 1
Question: Do the Cozy Comfort Pullover Set have side pockets?
Reference: Yes, the Cozy Comfort Pullover Set has side pockets.
Prediction: Yes, the Cozy Comfort Pullover Set does have side pockets. The description mentions that the pull-on pants feature "side pockets and a modern slim leg."
Grade: CORRECT
Reasoning: The model answer correctly states that the Cozy Comfort Pullover Set has side pockets, which aligns with the reference answer. The additional details about the pants' features do not contradict the core information and provide extra context.
Example 2
Question: What materials are used in the Ultra-Loft Down Hoodie?
Reference: I should answer using the materials listed in the catalog entry for the Ultra-Loft Down Hoodie.
Prediction: The Ultra-Loft Down Hoodie, as per the catalog, is not explicitly listed. However, based on the context provided, there are several jackets with similar names and features. If you're referring to the "Ultra-Loft 850 Big Baffle Hooded Jac